# Benchmarks - JavaScript

The one JavaScript example from [docs/benchmarks.md](https://platob.github.io/yggdryl/benchmarks/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

### The pipeline those numbers measure

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const zlib = require('node:zlib')
const { IOBase } = require('yggdryl')

const pattern =
  '^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}\\S*' +
  ' \\[(?<level>[^\\]]+)\\] \\[(?<logger>[^\\]]+)\\]' +
  ' \\[(?<thread_id>\\d+)\\] took=(?<latency_us>\\d+)'

// Two rotated leaves; the second record of the first spans a stack trace.
const leaves = [
  '2024-02-01 10:00:00.000000 [ii] [engine] [3] took=120 fill 100 SYMB-0001\n' +
    '2024-02-01 10:00:01.000000 [ee] [engine] [4] took=980 fill 101 SYMB-0002\n' +
    '    at engine::match(order.rs:118)\n' +
    '    at engine::step(order.rs:64)\n' +
    '2024-02-01 10:00:02.000000 [ww] [router] [5] took=240 fill 102 SYMB-0003\n',
  '2024-02-01 10:00:03.000000 [ee] [ledger] [6] took=770 fill 103 SYMB-0004\n' +
    '2024-02-01 10:00:04.000000 [ii] [feed] [7] took=100 fill 104 SYMB-0005\n',
]
const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
leaves.forEach((text, index) => {
  fs.writeFileSync(path.join(root, `app-${index}.log.gz`), zlib.gzipSync(Buffer.from(text)))
})

let rows = 0
let errors = 0
let traced = 0
let latency = 0n // an int64 column reads as BigInt

for (const batch of new IOBase(root).readArrowLines(pattern)) {
  const level = batch.getChild('level')
  const took = batch.getChild('latency_us')
  const spans = batch.getChild('lines')
  rows += batch.numRows
  for (let row = 0; row < batch.numRows; row += 1) {
    if (spans.get(row) > 1) traced += 1
    if (level.get(row) === 'ee') {
      errors += 1
      latency += took.get(row) // no Number() round trip
    }
  }
}

assert.deepEqual([rows, errors, traced], [5, 2, 1])
assert.equal(latency, 1750n)

fs.rmSync(root, { recursive: true, force: true })